In [6]:
# ! pip3 install requests bs4

In [7]:
import requests
from bs4 import BeautifulSoup
from typing import List, Optional,Dict
import datetime
import os
import json

In [8]:
def get_todays_date() -> str:
    """Return today's date in YYYY-MM-DD format."""
    return datetime.datetime.today().strftime('%Y-%m-%d')

def get_unique_paper_urls(k) -> Optional[List[str]]:
    """
    Fetch unique paper URLs from Hugging Face for today's date.

    Returns:
        Optional[List[str]]: A list of unique paper URLs, or None if the request fails.
    """
    BASE_URL = "https://huggingface.co"
    url = f"{BASE_URL}/papers?date={get_todays_date()}"
    
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raises an error for bad responses
    except requests.RequestException as e:
        print(f"Failed to retrieve data: {e}")
        return None

    soup = BeautifulSoup(response.text, 'html.parser')
    paper_links = soup.find_all('a', href=True)
    unique_urls = set([
        f"{BASE_URL}{link['href']}" for link in paper_links if "/papers/" in link['href'] and not "community" in link['href']
    ])

    return list(unique_urls)[:k]

In [9]:
unique_urls = get_unique_paper_urls(5)

In [10]:
def fetch_paper_details_and_download_pdf(url: str, save_directory: str) -> Optional[Dict[str, str]]:
    """
    Fetch details of a research paper from the provided Hugging Face URL and download the PDF.

    Parameters:
    url (str): The URL of the Hugging Face paper page.
    save_directory (str): The directory where the PDF will be saved.

    Returns:
    Optional[Dict[str, str]]: A dictionary containing the title, authors, and abstract of the paper,
                                or None if the fetching fails.
    """
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                          '(KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'
        }
        response = requests.get(url, headers=headers)
        response.raise_for_status()  # Raises an HTTPError for bad responses
        soup = BeautifulSoup(response.text, 'html.parser')

        # Extract title
        title_element = soup.find('h1')
        title = title_element.get_text(strip=True) if title_element else "Unknown Title"

        # Extract authors
        authors = [author.get_text(strip=True) for author in soup.select('div.author a')]
        authors_list = ', '.join(authors) if authors else "Unknown Authors"

        # Extract abstract
        abstract_element = soup.find('h2', string='Abstract')
        abstract = abstract_element.find_next('p').get_text(strip=True) if abstract_element else "No abstract available"

        # Find the PDF link
        pdf_link_element = None
        for a_tag in soup.find_all('a', class_='btn inline-flex h-9 items-center'):
            if 'View PDF' in a_tag.get_text():
                pdf_link_element = a_tag
                break
        if pdf_link_element and 'href' in pdf_link_element.attrs:
            pdf_link = pdf_link_element['href']
        else:
            raise ValueError("PDF link not found")

        # Download the PDF
        pdf_response = requests.get(pdf_link, headers=headers)
        pdf_response.raise_for_status()  # Raises an HTTPError for bad responses

        # Ensure the save directory exists
        os.makedirs(save_directory, exist_ok=True)

        # Create a filename for the PDF
        pdf_filename = os.path.join(save_directory, f"{title.replace(' ', '_')}.pdf")

        # Write the PDF to a file
        with open(pdf_filename, 'wb') as pdf_file:
            pdf_file.write(pdf_response.content)

        print(f"Downloaded PDF: {pdf_filename}")

        return {
            'title': title,
            'authors': authors_list,
            'abstract': abstract,
            'url':url
        }

    except requests.RequestException as e:
        print(f"Error fetching paper details or PDF: {e}")
        return None
    except ValueError as e:
        print(e)
        return None
    except Exception as e:
        print(f"An error occurred while processing the page: {e}")
        return None

In [11]:
save_directory = f'../papers2/{get_todays_date()}'  # Specify your directory here
os.makedirs(save_directory, exist_ok=True)  # Create the directory if it doesn't exist
paper_details_list = []
for url in unique_urls:
	paper_details = fetch_paper_details_and_download_pdf(url, save_directory)
	if paper_details:
		paper_details_list.append(paper_details)
with open(f"./papers2_{get_todays_date()}.json","w") as json_file:
    json.dump(paper_details_list,json_file)  # Save the list of papers to a JSON file

Downloaded PDF: ../papers2/2025-01-24/O1-Pruner:_Length-Harmonizing_Fine-Tuning_for_O1-Like_Reasoning_Pruning.pdf
Downloaded PDF: ../papers2/2025-01-24/Fast3R:_Towards_3D_Reconstruction_of_1000+_Images_in_One_Forward_Pass.pdf
Downloaded PDF: ../papers2/2025-01-24/Autonomy-of-Experts_Models.pdf
Downloaded PDF: ../papers2/2025-01-24/Pairwise_RM:_Perform_Best-of-N_Sampling_with_Knockout_Tournament.pdf
Downloaded PDF: ../papers2/2025-01-24/IntellAgent:_A_Multi-Agent_Framework_for_Evaluating_Conversational_AI
__Systems.pdf
